In [ ]:
%pip install -q zoomy-core

# Shallow Water Tutorial — SWE → SystemModel → NumericalSystemModel → C++ → Solve

This standalone Pyodide notebook walks the canonical Zoomy modeling stack on
the 1-D shallow-water equations:

1. Build SWE as a `Model`.
2. Freeze it as a `SystemModel` (operator-form sibling).
3. Bundle it into a `NumericalSystemModel` (`riemann`, `reconstruction`,
   `diffusion`, `regularization`).
4. Print C++ headers via the codegen printers.
5. Solve with the built-in NumPy `HyperbolicSolver` and the HLL Riemann.
6. Plot `h(x)` and `hu(x)` with matplotlib.

## Imports

In [ ]:
import numpy as np
from sympy import Matrix
import matplotlib.pyplot as plt
import pytest

from zoomy_core.fvm.solver_numpy import HyperbolicSolver, Settings
import zoomy_core.fvm.timestepping as timestepping
from zoomy_core.model.basemodel import Model
import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
from zoomy_core.mesh import BaseMesh
from zoomy_core.misc.misc import Zstruct

## 1. Shallow Water as a `Model`

Conservative 1-D SWE with state $Q = (h,\, hu)$ and flux
$F(Q) = (hu,\; hu^2/h + \tfrac{1}{2} g h^2)$.
Source is zero (frictionless, flat bed).

In [ ]:
class SWE(Model):
    dimension = 1
    # `h` declared positive so the symbolic eigenvalues simplify to the
    # textbook  u ± sqrt(g·h)  form; matches the assumption the solver
    # makes on the wet state.
    variables = {"h": (None, "positive"), "hu": None}
    parameters = {"g": (9.81, "positive")}

    def flux(self):
        # `self.parameters.g` is the *numeric* value (9.81) used for
        # runtime substitution.  For symbolic construction we want the
        # Symbol — that lives on `self._parameter_symbols`.
        v = self.variables
        p = self._parameter_symbols
        F = Matrix.zeros(self.n_variables, self.dimension)
        F[0, 0] = v.hu
        F[1, 0] = v.hu**2 / v.h + 0.5 * p.g * v.h**2
        return F

## 2. Freeze into a `SystemModel`

`Model` and `SystemModel` are **siblings, not parent/child**: same
operator surface (`flux`, `eigenvalues`, `nonconservative_matrix`, …),
different internals.  `SystemModel.from_model(m)` walks the model's
operator API once and stores the resulting matrices; subsequent calls
are pure data access.

In [ ]:
from zoomy_core.model.models.system_model import SystemModel

# A throw-away instance with stub BC/IC — only the symbolic operators
# matter for the demo.
_demo_bcs = BC.BoundaryConditions(
    [BC.Extrapolation(tag="left"), BC.Extrapolation(tag="right")]
)
_demo_ic = IC.UserFunction(lambda x: np.array([1.0, 0.0]))
model_demo = SWE(boundary_conditions=_demo_bcs, initial_conditions=_demo_ic)

sm = SystemModel.from_model(model_demo)
print("state         :", list(sm.state))
print("flux          :")
print(sm.flux)
print("eigenvalues   :", sm.eigenvalues)

## 3. Bundle into a `NumericalSystemModel`

`NumericalSystemModel` is the **numerical sibling of `SystemModel`** —
a configuration bundle (`riemann`, `reconstruction`, `diffusion`,
`regularization`, `lsq_degree`) that ties a `SystemModel` to a concrete
discretization.  The `HyperbolicSolver` consumes an NSM; if you hand it
a plain `Model`, it auto-promotes via `SystemModel.from_model` followed
by `NumericalSystemModel.from_system_model`.

The `riemann` field carries a **class**, not an instance.  Instantiate
it on the SystemModel to obtain the symbolic numerics (`Q_minus / Q_plus`,
`flux_minus / flux_plus`, `source_term`) that the codegen printers and
the solver lambdifier consume.

In [ ]:
from zoomy_core.numerics import NumericalSystemModel
from zoomy_core.fvm.riemann_solvers import HLL

nsm = NumericalSystemModel.from_system_model(sm, riemann=HLL)
print("riemann       :", nsm.riemann.__name__)
print("reconstruction:", nsm.reconstruction)
print("diffusion     :", nsm.diffusion)
print("regularization:", nsm.regularization)

# Instantiate the Riemann numerics for the codegen demo below.
num = nsm.riemann(nsm.sm)
print()
print("Q_minus       :", list(num.variables_minus))
print("flux_minus    :", list(num.flux_minus))
print("source_term   :", list(num.source_term))

## 4. Print to C++

The codegen printers turn a `Model` (or `Numerics`) into a portable
C++ header.  In the browser we use `.create_code()` for the string
form rather than `.write_code(...)` (the latter touches the
filesystem).

In [ ]:
from zoomy_core.transformation.to_c import CppModel, CppNumerics

model_code = CppModel(model_demo).create_code()
preview = "\n".join(model_code.splitlines()[:60])
print(preview)
print(f"\n... ({len(model_code.splitlines())} lines total — see next cell)")

In [ ]:
print(model_code)

In [ ]:
num_code = CppNumerics(num).create_code()
preview = "\n".join(num_code.splitlines()[:60])
print(preview)
print(f"\n... ({len(num_code.splitlines())} lines total — see next cell)")

In [ ]:
print(num_code)

## 5. Solve with the NumPy `HyperbolicSolver`

Classic dam-break: $h_0 = 0.005$ for $x < 5$, $0.001$ otherwise;
$hu_0 = 0$.  Extrapolation BCs left and right.

In [ ]:
bcs = BC.BoundaryConditions(
    [
        BC.Extrapolation(tag="left"),
        BC.Extrapolation(tag="right"),
    ]
)

def dam_break_ic(x):
    Q = np.zeros(2, dtype=float)
    Q[0] = np.where(x[0] < 5.0, 0.005, 0.001)
    return Q

ic = IC.UserFunction(dam_break_ic)

model1d = SWE(
    boundary_conditions=bcs,
    initial_conditions=ic,
)

In [ ]:
mesh1d = BaseMesh.create_1d(domain=(0.0, 10.0), n_inner_cells=200)

In [ ]:
settings = Settings(
    name="ShallowWater",
    output=Zstruct(
        directory="outputs/shallow_water_1d",
        filename="swe",
        clean_directory=True,
        snapshots=10,
    ),
)

# Use the same HLL Riemann choice as the symbolic demo above.
nsm1d = NumericalSystemModel.from_system_model(
    SystemModel.from_model(model1d), riemann=HLL
)

solver = HyperbolicSolver(
    time_end=6.0,
    settings=settings,
    compute_dt=timestepping.adaptive(CFL=0.95),
)
Qnew, Qaux = solver.solve(mesh1d, nsm1d, write_output=False)

## 6. Plot the result

Plot the final water depth $h(x)$ and momentum $hu(x)$ directly from
the in-memory state — no VTK round-trip.

In [ ]:
n = mesh1d.n_inner_cells
xc = mesh1d.cell_centers_computed()[0, :n]
h  = Qnew[0, :n]
hu = Qnew[1, :n]

fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
axes[0].plot(xc, h);  axes[0].set_xlabel("x"); axes[0].set_title("h(x, t_end)")
axes[1].plot(xc, hu); axes[1].set_xlabel("x"); axes[1].set_title("hu(x, t_end)")
fig

In [ ]:
@pytest.mark.nbworking
def test_working():
    assert True